In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [3]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

model = ChatOllama(model="llama3.2", temperature=0.3)

agent = create_agent(
    model=model,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [4]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [5]:
from pprint import pprint, PrettyPrinter

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'address': 'your_email_address',
                                                                  'body': 'Your '
                                                                          'reply: '
                                                                          'This '
                                                                          'is '
                                                                          'a '
                                                                          'response '
                                                                          'to '
                                                                          'your '
                                                                          'email.'},
                                                         'description': 'Tool '
                                                                        'executio

In [6]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'address': 'your_email_address', 'body': 'Your reply: This is a response to your email.'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'address': 'your_email_address', 'body': 'Your reply: This is a response to your email.'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='0cf9db8c710c226041ad40f9682a5b46')]


In [7]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Your reply: This is a response to your email.


## Approve

In [8]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='5371cda1-e0af-423e-ba7e-0715ab6a86f3'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-09-19T12:34:01.847056464Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6401087442, 'load_duration': 5725675518, 'prompt_eval_count': 197, 'prompt_eval_duration': 67381014, 'eval_count': 54, 'eval_duration': 570586332, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a0b9a8-b09f-71d2-97f4-307cfb97495c-0', tool_calls=[{'name': 'read_email', 'args': {'address': 'your_email_address'}, 'id': '323aa18e-4986-47ec-989d-fef239f2014d', 'type': 'tool_call'}, {'name': 'send_email', 'args': {'address': 'your

## Reject

In [9]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='5371cda1-e0af-423e-ba7e-0715ab6a86f3'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-09-19T12:34:01.847056464Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6401087442, 'load_duration': 5725675518, 'prompt_eval_count': 197, 'prompt_eval_duration': 67381014, 'eval_count': 54, 'eval_duration': 570586332, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a0b9a8-b09f-71d2-97f4-307cfb97495c-0', tool_calls=[{'name': 'read_email', 'args': {'address': 'your_email_address'}, 'id': '323aa18e-4986-47ec-989d-fef239f2014d', 'type': 'tool_call'}, {'name': 'send_email', 'args': {'address': 'your

In [10]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

KeyError: '__interrupt__'

## Edit

In [11]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response, width=120)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.",
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='5371cda1-e0af-423e-ba7e-0715ab6a86f3'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-09-19T12:34:01.847056464Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6401087442, 'load_duration': 5725675518, 'prompt_eval_count': 197, 'prompt_eval_duration': 67381014, 'eval_count': 54, 'eval_duration': 570586332, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a0b9a8-b09f-71d2-97f4-307cfb97495c-0', tool_calls=[{'name': 'read_email', 'args': {'address': 'your_email_address'}, 'id': '323aa18e-4986-47ec-989d-fef239f2014d', 'type': 'tool_call'}, {'name': 'send_email', 'args': {'address': 'your_email_addres